# cALorie Notebook 41 — RepCount-A intake v2, shard 3 of 7

**Runtime: CPU. Run all.** Output: `calorie_repcount_intake_v2_shard3.zip`.

Evidence only. No labels, training, model scoring, or locked-benchmark access.

- Shard 3 of the 946 remaining RepCount-A clips (`ml/data/repcount_intake_v2/full_selection.csv`).
- For every clip: 15-FPS MediaPipe pose inside the annotated repetition bounds, a 12-frame contact
  sheet, and 3 scene thumbnails. The code is Notebook 40's, which ran on 94 real clips.
- Videos are read one at a time from the revision-pinned Hugging Face Lance mirror and deleted
  right after processing. Raw video never enters the ZIP.
- A clip that fails is recorded in `source_audit.csv` instead of stopping the run.

Plan: `ml/data/repcount_intake_v2/INTAKE_PLAN.md`; locked pilot results: `PILOT_RESULTS.md`.


In [ ]:
%pip install -q --pre --upgrade --extra-index-url https://pypi.fury.io/lance-format pylance mediapipe==0.10.35 opencv-python-headless==4.11.0.86 pandas==2.2.3


In [ ]:
import hashlib, importlib.metadata, json, math, os, shutil, time, traceback, urllib.request, zipfile
from pathlib import Path

import cv2
import lance
import mediapipe as mp
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    from IPython.display import display
except ImportError:  # local dry run outside IPython
    display = print

# Local dry-run switches. Colab leaves all of these unset.
DRY_RUN = os.environ.get("CALORIE_DRY_RUN") == "1"
LOCAL_REPO = os.environ.get("CALORIE_LOCAL_REPO")
WORK = Path(os.environ.get("CALORIE_WORK", "/content")) / "calorie_repcount_intake_v2_shard3_work"

PLAN_COMMIT = "f69a9c4a3d6ae565aed93c2e18ebfaae80a5cc17"
SELECTION_PATH = "ml/data/repcount_intake_v2/full_selection.csv"
SELECTION_SHA256 = "0ce07188a8755d45c1bbea2b3b4fdbf5ce2892432e0e2ee9c42f0d1665f135f7"
SHARD = 3
LANCE_REVISION = "ae2e0b0496124d900d4e54a8e92f7ec15a292afd"
TARGET_FPS = 15.0
SHEET_TILES = (4, 3)  # columns, rows
SCENE_POSITIONS = (0.15, 0.50, 0.85)
SCENE_MAX_SIDE = 256
MIN_POSE_DETECTION = 0.8

if WORK.exists():
    shutil.rmtree(WORK)
INPUTS, VIDEOS, POSE, SHEETS, SCENE, REPORT = [WORK / x for x in ("inputs", "videos", "pose", "sheets", "scene", "report")]
for path in (INPUTS, VIDEOS, POSE, SHEETS, SCENE, REPORT):
    path.mkdir(parents=True, exist_ok=True)
print("Dry run:", DRY_RUN, "| PyLance:", importlib.metadata.version("pylance"), "| MediaPipe:", mp.__version__, "| OpenCV:", cv2.__version__)


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


target = INPUTS / "full_selection.csv"
if LOCAL_REPO:
    shutil.copy2(Path(LOCAL_REPO) / SELECTION_PATH, target)
else:
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/bariscelikk1/cALorie/{PLAN_COMMIT}/{SELECTION_PATH}", target)
assert sha256(target) == SELECTION_SHA256, "full_selection.csv changed; this notebook is pinned to the approved selection."
selection = pd.read_csv(target)
assert selection.video_id.is_unique
assert len(selection) == 946 and selection.shard.between(0, 7).all()
selection = selection[selection.shard == SHARD].assign(role="intake").reset_index(drop=True)
assert 110 <= len(selection) <= 125, len(selection)
if DRY_RUN:
    selection = selection.head(3)
display(selection.groupby("bucket").size())


## Match the selection to the pinned Lance mirror


In [ ]:
scalar_columns = ["video_id", "source_name", "action_type", "count", "cycle_bounds_json", "video_ext", "video_size_bytes"]
datasets, parts = {}, []
for split in ("train", "validation", "test"):
    uri = f"hf://datasets/lmms-lab-eval/repcounta-lance/data/{split}.lance"
    ds = lance.dataset(uri, storage_options={"hf_revision": LANCE_REVISION, "hf_enable_resolve_cache": "true"})
    frame = ds.to_table(columns=scalar_columns, with_row_id=True).to_pandas()
    frame["lance_split"] = split
    datasets[split] = ds
    parts.append(frame)
mirror = pd.concat(parts, ignore_index=True).rename(columns={"_rowid": "lance_row_id"})
assert mirror.video_id.is_unique
selected = selection.merge(mirror, on="video_id", how="left", validate="one_to_one")
assert selected.lance_row_id.notna().all(), selected[selected.lance_row_id.isna()].video_id.tolist()
assert selected.published_action.eq(selected.action_type).all()
assert selected.published_count.astype(int).eq(selected["count"].astype(int)).all()
print("Matched", len(selected), "rows; selected GB:", round(selected.video_size_bytes.sum() / 1e9, 2))


## Per-clip evidence

For each clip: read the blob, check its size, render the evidence, extract pose (pilot only),
then delete the video before the next clip.


In [ ]:
MODEL_PATH = WORK / "pose_landmarker_full.task"
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task"
urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
POSE_MODEL_SHA256 = sha256(MODEL_PATH)
KEEP_17 = [0, 2, 5, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
LEFT_SHOULDER, RIGHT_SHOULDER, LEFT_HIP, RIGHT_HIP = 5, 6, 11, 12


def parse_bounds(value):
    try:
        value = json.loads(value) if isinstance(value, str) else value
    except json.JSONDecodeError:
        return []
    numbers = []

    def walk(item):
        if isinstance(item, dict):
            for child in item.values():
                walk(child)
        elif isinstance(item, (list, tuple, np.ndarray)):
            for child in item:
                walk(child)
        elif isinstance(item, (int, float)) and np.isfinite(item):
            numbers.append(float(item))

    walk(value)
    return sorted(x for x in numbers if x >= 0)


def probe(path):
    cap = cv2.VideoCapture(str(path))
    assert cap.isOpened(), path
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    cap.release()
    assert fps > 0 and frames > 0 and width > 0 and height > 0, (path, fps, frames, width, height)
    return {"source_fps": fps, "source_frames": frames, "duration_s": frames / fps, "width": width, "height": height}


def read_frame(cap, time_s):
    cap.set(cv2.CAP_PROP_POS_MSEC, float(time_s) * 1000)
    ok, frame = cap.read()
    assert ok, time_s
    return frame


def fit(frame, width, height):
    scale = min(width / frame.shape[1], height / frame.shape[0])
    return cv2.resize(frame, (max(1, int(round(frame.shape[1] * scale))), max(1, int(round(frame.shape[0] * scale)))), interpolation=cv2.INTER_AREA)


def contact_sheet(path, video_id, start_s, end_s):
    columns, rows = SHEET_TILES
    cap = cv2.VideoCapture(str(path))
    sheet = np.zeros((rows * 205, columns * 320, 3), dtype=np.uint8)
    for index, time_s in enumerate(np.linspace(start_s, end_s, columns * rows)):
        resized = fit(read_frame(cap, time_s), 320, 180)
        tile = np.zeros((205, 320, 3), dtype=np.uint8)
        x0 = (320 - resized.shape[1]) // 2
        y0 = 25 + (180 - resized.shape[0]) // 2
        tile[y0:y0 + resized.shape[0], x0:x0 + resized.shape[1]] = resized
        cv2.putText(tile, f"{video_id} | {time_s:.2f}s", (7, 17), cv2.FONT_HERSHEY_SIMPLEX, 0.43, (0, 255, 255), 1, cv2.LINE_AA)
        sheet[(index // columns) * 205:(index // columns + 1) * 205, (index % columns) * 320:(index % columns + 1) * 320] = tile
    cap.release()
    output = SHEETS / f"{video_id}.jpg"
    assert cv2.imwrite(str(output), sheet, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return output


def scene_thumbnails(path, video_id, duration_s):
    cap = cv2.VideoCapture(str(path))
    outputs = []
    for index, position in enumerate(SCENE_POSITIONS):
        frame = fit(read_frame(cap, duration_s * position), SCENE_MAX_SIDE, SCENE_MAX_SIDE)
        output = SCENE / f"{video_id}_{index}.jpg"
        assert cv2.imwrite(str(output), frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
        outputs.append(output)
    cap.release()
    return outputs


def normalize_pose(frame):
    xyz = frame[:, :3].copy()
    hip = (xyz[LEFT_HIP] + xyz[RIGHT_HIP]) / 2
    shoulder = (xyz[LEFT_SHOULDER] + xyz[RIGHT_SHOULDER]) / 2
    scale = max(float(np.linalg.norm(xyz[LEFT_SHOULDER] - xyz[RIGHT_SHOULDER])), float(np.linalg.norm(shoulder - hip)), 1e-6)
    return np.concatenate([(xyz - hip) / scale, frame[:, 3:4]], axis=1)


def fill_missing(sequence):
    flat = sequence.reshape(len(sequence), -1)
    return pd.DataFrame(flat).interpolate(limit_direction="both").fillna(0).to_numpy().reshape(sequence.shape).astype(np.float32)


def extract_pose(path, start_frame, end_frame, source_fps):
    cap = cv2.VideoCapture(str(path))
    assert cap.isOpened()
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_frame))
    next_sample = start_frame / source_fps
    frame_index = int(start_frame)
    poses, timestamps, detected, raw = [], [], [], []
    options = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=mp.tasks.vision.RunningMode.VIDEO, num_poses=1,
        min_pose_detection_confidence=0.5, min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5, output_segmentation_masks=False,
    )
    with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
        while frame_index <= int(end_frame):
            ok, frame = cap.read()
            if not ok:
                break
            time_s = frame_index / source_fps
            frame_index += 1
            if time_s + 1e-9 < next_sample:
                continue
            next_sample += 1 / TARGET_FPS
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            result = landmarker.detect_for_video(image, int(round(time_s * 1000)))
            if result.pose_landmarks:
                points = result.pose_landmarks[0]
                pose = np.asarray([[points[i].x, points[i].y, points[i].z, float(points[i].visibility or 0)] for i in KEEP_17], dtype=np.float32)
                poses.append(normalize_pose(pose))
                raw.append(pose)
                detected.append(True)
            else:
                poses.append(np.full((17, 4), np.nan, dtype=np.float32))
                raw.append(np.full((17, 4), np.nan, dtype=np.float32))
                detected.append(False)
            timestamps.append(time_s)
    cap.release()
    assert poses, "no frames sampled"
    # Unnormalized image-space landmarks are kept for orientation rules (e.g. supine vs prone).
    return fill_missing(np.stack(poses)), fill_missing(np.stack(raw)), np.asarray(timestamps, np.float32), np.asarray(detected, bool)


FETCH_ATTEMPTS = 6


def fetch_blob(row):
    # Hugging Face sometimes fails a run of requests temporarily; wait and reopen before giving up.
    for attempt in range(FETCH_ATTEMPTS):
        try:
            blob = datasets[row.lance_split].take_blobs("video_blob", ids=[int(row.lance_row_id)])[0]
            payload = blob.read()
            return payload.to_pybytes() if hasattr(payload, "to_pybytes") else payload
        except OSError as error:
            if attempt == FETCH_ATTEMPTS - 1:
                raise
            wait_s = min(15 * 2 ** attempt, 240)
            print(f"{row.video_id}: fetch attempt {attempt + 1} failed ({type(error).__name__}); retrying in {wait_s}s")
            time.sleep(wait_s)
            uri = f"hf://datasets/lmms-lab-eval/repcounta-lance/data/{row.lance_split}.lance"
            datasets[row.lance_split] = lance.dataset(uri, storage_options={"hf_revision": LANCE_REVISION, "hf_enable_resolve_cache": "true"})


audit_rows = []
for row in tqdm(selected.itertuples(index=False), total=len(selected), desc="Clips"):
    record = {"video_id": row.video_id, "role": row.role, "status": "ok", "error": ""}
    extension = str(row.video_ext or ".mp4")
    video_path = VIDEOS / f"{row.video_id}{extension if extension.startswith('.') else '.' + extension}"
    try:
        video_path.write_bytes(fetch_blob(row))
        assert video_path.stat().st_size == int(row.video_size_bytes), "blob size mismatch"
        record["video_sha256"] = sha256(video_path)
        info = probe(video_path)
        record.update(info)
        bounds = parse_bounds(row.cycle_bounds_json)
        start_frame = max(0, int(math.floor(min(bounds)))) if bounds else 0
        end_frame = min(info["source_frames"] - 1, int(math.ceil(max(bounds)))) if bounds else info["source_frames"] - 1
        assert end_frame > start_frame, ("bad bounds", bounds)
        record.update({"cycle_boundary_count": len(bounds), "annotated_start_frame": start_frame, "annotated_end_frame": end_frame,
                       "annotated_start_s": start_frame / info["source_fps"], "annotated_end_s": end_frame / info["source_fps"]})
        scene_thumbnails(video_path, row.video_id, info["duration_s"])
        if row.role == "intake":
            contact_sheet(video_path, row.video_id, record["annotated_start_s"], record["annotated_end_s"])
            landmarks, image_landmarks, timestamps, detected = extract_pose(video_path, start_frame, end_frame, info["source_fps"])
            np.savez_compressed(POSE / f"{row.video_id}.npz", landmarks=landmarks, image_landmarks=image_landmarks,
                                timestamps_s=timestamps, detected=detected, target_fps=np.float32(TARGET_FPS),
                                source_fps=np.float32(info["source_fps"]), video_id=row.video_id)
            record.update({"sampled_frames": len(landmarks), "pose_detection_rate": float(detected.mean())})
            if detected.mean() < MIN_POSE_DETECTION:
                record["status"] = "low_pose_detection"
    except Exception as error:  # record and continue; the audit reports every failure
        record.update({"status": "error", "error": f"{type(error).__name__}: {error}"})
        traceback.print_exc()
    finally:
        video_path.unlink(missing_ok=True)  # licensed raw media never persists
    audit_rows.append(record)

source_audit = selected.drop(columns=["cycle_bounds_json"]).merge(pd.DataFrame(audit_rows), on=["video_id", "role"], validate="one_to_one")
source_audit.to_csv(REPORT / "source_audit.csv", index=False)
assert not any(VIDEOS.iterdir()), "Raw video left on disk."
display(source_audit.groupby(["role", "status"]).size())
display(source_audit[source_audit.status != "ok"][["video_id", "role", "status", "error"]])


## Package and download


In [ ]:
failed = int((source_audit.status == "error").sum())


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


run_metadata = {
    "notebook": "41_colab_repcount_intake_v2_shard3.ipynb",
    "experiment": "repcount_intake_v2_full", "shard": SHARD,
    "plan": "ml/data/repcount_intake_v2/INTAKE_PLAN.md",
    "plan_commit": PLAN_COMMIT, "selection_sha256": SELECTION_SHA256,
    "repcount_lance_revision": LANCE_REVISION, "dry_run": DRY_RUN,
    "pose_model_url": MODEL_URL, "pose_model_sha256": POSE_MODEL_SHA256,
    "packages": {name: package_version(name) for name in ("pylance", "mediapipe", "opencv-python-headless", "pandas", "numpy")},
    "opencv_runtime": cv2.__version__,
    "target_fps": TARGET_FPS, "rows": int(len(source_audit)),
    "intake_rows": int((source_audit.role == "intake").sum()), "errors": failed,
    "raw_video_packaged": False, "labels_created": False, "training_performed": False,
    "model_scored": False, "locked_benchmark_read": False,
}
(REPORT / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2) + "\n")
(REPORT / "README.md").write_text(
    f"# cALorie RepCount-A intake v2 evidence, shard {SHARD}\n\n"
    + ("DRY RUN: a 3-clip local test, not the shard.\n\n" if DRY_RUN else "")
    + f"Pose, contact sheets, and scene thumbnails for {len(source_audit)} clips. No labels, training, model scoring, or locked-benchmark access. Raw video is excluded. "
    f"Clips with errors: {failed} (see source_audit.csv).\n"
)
shutil.copy2(INPUTS / "full_selection.csv", REPORT / "full_selection.csv")
for folder in (POSE, SHEETS, SCENE):
    shutil.copytree(folder, REPORT / folder.name, dirs_exist_ok=True)
assert not list(REPORT.rglob("*.mp4")), "Raw video must not be packaged."
checksum_lines = [f"{sha256(path)}  {path.relative_to(REPORT)}" for path in sorted(p for p in REPORT.rglob("*") if p.is_file())]
(REPORT / "SHA256SUMS.txt").write_text("\n".join(checksum_lines) + "\n")
zip_path = WORK.parent / ("calorie_repcount_intake_v2_shard3" + ("_DRYRUN" if DRY_RUN else "") + ".zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(p for p in REPORT.rglob("*") if p.is_file()):
        archive.write(path, Path("report") / path.relative_to(REPORT))
print("Created:", zip_path, round(zip_path.stat().st_size / 1e6, 1), "MB | clips with errors:", failed)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Not on Colab; ZIP left at", zip_path)


## Return the ZIP

Send back `calorie_repcount_intake_v2_shard3.zip` unchanged. A few clip errors are fine; they are listed in `source_audit.csv`.
